# EE533 Final Project: Circuit-Informed Integrate-and-Fire Neuron

This notebook starts the Spring 2026 EE533 final project workflow from the Cadence neuron circuit.

Current status:
- Build a Python neuron model matched to the Cadence circuit
- Reproduce the measured firing-rate behavior
- Validate constant-current and pulsed-current behavior
- Prepare the neuron model for later SNN integration on MNIST

The present notebook uses the measured values visible from the Cadence screenshots and one measured operating point near `100 nA -> 9.994 kHz`. Replace the placeholder values with exported Cadence sweep data as more measurements become available.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from circuit_if_neuron import (
    CadenceNeuronParameters,
    analytical_rate_hz,
    calibrate_current_gain_from_point,
    constant_current_waveform,
    fi_curve,
    measure_firing_rate,
    nearest_neighbor_timing_error_s,
    pulsed_current_waveform,
    simulate_neuron,
)

plt.style.use('ggplot')
np.set_printoptions(suppress=True)


ModuleNotFoundError: No module named 'pandas'

## 1. Cadence-Informed Parameters

These values come from the schematic and waveform screenshots you provided:
- `cfb = 3.757 pF`
- `cm = 24.5 pF`
- `iin = 100 nA`
- `irst = 2 uA`
- `wn = 10 um`, `wp = 20 um`
- `ln = 600 nm`, `lp = 600 nm`
- measured frequency at `iin = 100 nA` is about `9.994 kHz`

The threshold and reset voltages below are estimated from the waveform screenshot and should be refined with exported Cadence traces.


In [ ]:
base_params = CadenceNeuronParameters(
    vdd=3.2,
    c_mem=24.5e-12,
    c_fb=3.757e-12,
    v_reset=1.12,
    v_threshold=1.83,
    refractory_s=2e-6,
    pulse_width_s=2e-6,
    leak_conductance_s=0.0,
    feedback_cap_scale=1.0,
    current_gain=1.0,
)

measured_current_a = 100e-9
measured_rate_hz = 9.994e3
fitted_gain = calibrate_current_gain_from_point(base_params, measured_current_a, measured_rate_hz)
params = base_params.with_updates(current_gain=fitted_gain)

summary = pd.Series(
    {
        'vdd_V': params.vdd,
        'c_mem_F': params.c_mem,
        'c_fb_F': params.c_fb,
        'v_reset_V': params.v_reset,
        'v_threshold_V': params.v_threshold,
        'refractory_s': params.refractory_s,
        'pulse_width_s': params.pulse_width_s,
        'total_capacitance_F': params.total_capacitance,
        'delta_v_V': params.delta_v,
        'fitted_current_gain': params.current_gain,
    }
)
summary


## 2. Analytical f-I Characteristic

For a constant input current, the simplified model follows:

\[
T = 
rac{C_{eff}(V_{th}-V_{reset})}{g_I I_{in}} + T_{ref}, \qquad f = 
rac{1}{T}
\]

where `g_I` is a fitted effective-current gain that absorbs non-idealities not yet explicitly modeled.


In [ ]:
currents_nA = np.logspace(0, 2, 200)  # 1 nA to 100 nA
currents_a = currents_nA * 1e-9
rates_hz = fi_curve(params, currents_a)

plt.figure(figsize=(7, 4))
plt.semilogx(currents_nA, rates_hz, linewidth=2, label='Python model')
plt.scatter([measured_current_a * 1e9], [measured_rate_hz], color='black', zorder=5, label='Measured point')
plt.xlabel('Input current (nA)')
plt.ylabel('Firing rate (Hz)')
plt.title('Circuit-Informed f-I Characteristic')
plt.legend()
plt.tight_layout()
plt.show()

pd.DataFrame(
    {
        'current_nA': [1, 2, 5, 10, 20, 50, 100],
        'predicted_rate_Hz': [analytical_rate_hz(params, x * 1e-9) for x in [1, 2, 5, 10, 20, 50, 100]],
    }
)


## 3. Constant-Current Validation

This simulates the same constant-current case used for the measured Cadence point. The main outputs are:
- membrane voltage `vmem`
- output pulse `vout`
- measured spike rate from the generated spikes


In [ ]:
dt_s = 0.1e-6
duration_s = 2.0e-3
constant_result = simulate_neuron(
    params=params,
    duration_s=duration_s,
    dt_s=dt_s,
    input_current=constant_current_waveform(measured_current_a),
)

simulated_rate_hz = measure_firing_rate(constant_result['spike_times_s'], duration_s)
print(f'Simulated firing rate: {simulated_rate_hz:,.2f} Hz')
print(f'Measured firing rate : {measured_rate_hz:,.2f} Hz')
print(f'Rate error           : {simulated_rate_hz - measured_rate_hz:,.2f} Hz')


In [ ]:
time_ms = constant_result['time_s'] * 1e3
fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True)

axes[0].plot(time_ms, constant_result['input_current_a'] * 1e9, color='tab:blue')
axes[0].set_ylabel('Iin (nA)')
axes[0].set_title('Constant-Current Validation')

axes[1].plot(time_ms, constant_result['vmem_v'], color='tab:green')
axes[1].axhline(params.v_threshold, linestyle='--', color='black', linewidth=1, label='threshold')
axes[1].axhline(params.v_reset, linestyle=':', color='black', linewidth=1, label='reset')
axes[1].set_ylabel('Vmem (V)')
axes[1].legend(loc='upper right')

axes[2].plot(time_ms, constant_result['vout_v'], color='tab:red')
axes[2].set_ylabel('Vout (V)')
axes[2].set_xlabel('Time (ms)')

plt.tight_layout()
plt.show()


## 4. Pulsed-Current Validation

Task 3 also requires validation with pulsed current. This section builds a repeatable test waveform. Once you export Cadence transient data for the same pulsed input, the timing agreement metrics below can be filled with real circuit data.


In [ ]:
pulse_result = simulate_neuron(
    params=params,
    duration_s=3.0e-3,
    dt_s=dt_s,
    input_current=pulsed_current_waveform(
        baseline_a=5e-9,
        pulse_a=60e-9,
        start_s=0.3e-3,
        width_s=0.25e-3,
        period_s=0.6e-3,
    ),
)

fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
time_ms = pulse_result['time_s'] * 1e3
axes[0].plot(time_ms, pulse_result['input_current_a'] * 1e9, color='tab:blue')
axes[0].set_ylabel('Iin (nA)')
axes[0].set_title('Pulsed-Current Validation')
axes[1].plot(time_ms, pulse_result['vmem_v'], color='tab:green')
axes[1].set_ylabel('Vmem (V)')
axes[2].plot(time_ms, pulse_result['vout_v'], color='tab:red')
axes[2].set_ylabel('Vout (V)')
axes[2].set_xlabel('Time (ms)')
plt.tight_layout()
plt.show()

print('Pulse-case spike count:', len(pulse_result['spike_times_s']))
print('First 10 spike times (ms):')
print(np.round(pulse_result['spike_times_s'][:10] * 1e3, 4))


## 5. Cadence CSV Import Hook

When you export Cadence results to CSV, put the file path into `csv_path` and update the column names if needed. This will allow direct circuit-vs-Python comparisons for:
- spike timing
- firing rate
- temporal alignment


In [ ]:
csv_path = None  # Example: 'cadence_exports/constant_current_100nA.csv'

if csv_path:
    cadence_df = pd.read_csv(csv_path)
    cadence_df.head()
else:
    print('No Cadence CSV loaded yet. Export transient data from Cadence and set csv_path.')


In [ ]:
# Example comparison stub once Cadence spike times are available.
# Replace cadence_spikes_s with spike times extracted from Cadence vout.

cadence_spikes_s = np.array([])
python_spikes_s = pulse_result['spike_times_s']

if len(cadence_spikes_s) > 0:
    timing_error_s = nearest_neighbor_timing_error_s(cadence_spikes_s, python_spikes_s)
    print(f'Mean nearest-neighbor timing error: {timing_error_s * 1e6:.3f} us')
else:
    print('Cadence spike times not loaded yet.')


## 6. Next Steps for the Full Assignment

This notebook now covers the first part of Task 2 and Task 3. The remaining work should proceed in this order:
1. Replace the single measured point with a full Cadence `f-I` sweep from `1 nA` to `100 nA`
2. Tune `v_reset`, `v_threshold`, leak, and `current_gain` against that sweep
3. Export pulsed-current transient data and compute spike-time agreement metrics
4. Wrap the neuron in the SNN training pipeline for MNIST
5. Compare default-neuron and circuit-informed-neuron accuracy across resolution, time-step, training, and quantization sweeps
